# SpikingBrain-7B — Google Colab Demo

**Requisiti GPU**: T4 (15 GB VRAM) o superiore  
**Modello**: Chat 7B-SFT da ModelScope (`Panyuqi/V1-7B-sft-s3-reasoning`)  
**Paper**: [arXiv:2509.05276](https://arxiv.org/abs/2509.05276)

> Assicurati di avere il runtime GPU abilitato:  
> `Runtime → Change runtime type → T4 GPU`

## 1. Verifica GPU e Memoria Disponibile

In [ ]:
import subprocess

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU non trovata! Vai su Runtime → Change runtime type → T4 GPU"
    )

gpu_name = torch.cuda.get_device_name(0)
vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name}")
print(f"VRAM totale: {vram_total:.1f} GB")

if vram_total < 14:
    print("ATTENZIONE: VRAM < 14 GB. Il modello full bfloat16 potrebbe non entrare.")
    print("Considera Colab Pro (A100) oppure usa la versione quantizzata W8ASpike.")
else:
    print("OK: VRAM sufficiente per il modello 7B in bfloat16.")

## 2. Installazione Dipendenze

> Questa cella richiede circa **5-10 minuti** la prima volta (flash_attn compila da sorgente).

In [ ]:
import sys

# 1. Aggiorna pip
!pip install -q --upgrade pip

# 2. Libreria ModelScope (per scaricare il modello)
!pip install -q modelscope

# 3. Transformers alla versione richiesta
!pip install -q transformers==4.55.2

# 4. Flash-Linear-Attention (kernel GLA via Triton)
!pip install -q flash-linear-attention==0.1

# 5. Flash Attention 2 (sliding window attention)
#    --no-build-isolation usa torch/CUDA gia' presenti su Colab
!pip install -q flash-attn==2.7.3 --no-build-isolation

# 6. Dipendenze accessorie
!pip install -q scipy pyyaml decorator

print("Installazione completata!")

## 3. Clona il Repository (codice del modello)

In [ ]:
import os

REPO_DIR = "/content/SpikingBrain-7B"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/BICLab/SpikingBrain-7B.git {REPO_DIR}
else:
    print("Repository gia' presente, aggiorno...")
    !git -C {REPO_DIR} pull

# Aggiunge il codice del modello al path Python
sys.path.insert(0, f"{REPO_DIR}/hf_7B_model")
print(f"Repository pronto in: {REPO_DIR}")

## 4. Download del Modello da ModelScope

> Il modello SFT da 7B pesa circa **14 GB** — il download richiede alcuni minuti.

In [ ]:
from modelscope import snapshot_download

MODEL_ID = "Panyuqi/V1-7B-sft-s3-reasoning"  # Chat model (SFT)
MODEL_DIR = "/content/spikingbrain_7b_chat"

if not os.path.exists(MODEL_DIR):
    print(f"Download modello '{MODEL_ID}'...")
    model_path = snapshot_download(
        MODEL_ID,
        cache_dir=MODEL_DIR
    )
    print(f"Modello scaricato in: {model_path}")
else:
    import glob
    dirs = glob.glob(f"{MODEL_DIR}/**", recursive=False)
    model_path = dirs[0] if dirs else MODEL_DIR
    print(f"Modello gia' presente: {model_path}")

print("Download completato!")

## 5. Caricamento Modello e Tokenizer

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"Caricamento tokenizer da: {model_path}")
tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    padding_side='left',
    truncation_side='left',
    trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Tokenizer caricato.")

print(f"Caricamento modello (bfloat16, CUDA)...")
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16
).cuda()
model.eval()

# Memoria GPU usata dopo il caricamento
mem_used = torch.cuda.memory_allocated() / 1e9
mem_reserved = torch.cuda.memory_reserved() / 1e9
print(f"Modello caricato! VRAM usata: {mem_used:.1f} GB (riservata: {mem_reserved:.1f} GB)")
print(f"Parametri totali: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

## 6. Funzione di Generazione

In [ ]:
def generate_response(
    user_message: str,
    system_message: str = "You are a helpful assistant.",
    max_new_tokens: int = 512,
    temperature: float = 0.7,
    top_p: float = 0.9,
    repetition_penalty: float = 1.1,
) -> str:
    """Genera una risposta dal modello SpikingBrain-7B."""
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to("cuda")

    with torch.no_grad():
        generated_ids = model.generate(
            inputs.input_ids,
            max_new_tokens=max_new_tokens,
            use_cache=True,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_ids = generated_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True)

print("Funzione 'generate_response' pronta.")

## 7. Test Rapido

In [ ]:
prompt = "Explain in simple terms what a spiking neural network is and why it is interesting."

print(f"Prompt: {prompt}\n")
print("Generazione risposta...\n")

response = generate_response(prompt, max_new_tokens=256)

print("=" * 60)
print(response)
print("=" * 60)

## 8. Chat Interattiva

Esegui questa cella per una sessione di chat interattiva.  
Digita `exit` o `quit` per terminare.

In [ ]:
print("SpikingBrain-7B Chat — digita 'exit' per uscire\n")
print("-" * 60)

while True:
    user_input = input("Tu: ").strip()
    if not user_input:
        continue
    if user_input.lower() in ("exit", "quit"):
        print("Chat terminata.")
        break

    response = generate_response(user_input)
    print(f"\nSpikingBrain: {response}\n")
    print("-" * 60)

## 9. Benchmark Velocita' (opzionale)

In [ ]:
import time

prompt = "Describe the architecture of a hybrid transformer model that combines linear and standard attention."
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": prompt},
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([text], return_tensors="pt").to("cuda")

MAX_NEW_TOKENS = 200

torch.cuda.synchronize()
start = time.time()

with torch.no_grad():
    generated_ids = model.generate(
        inputs.input_ids,
        max_new_tokens=MAX_NEW_TOKENS,
        use_cache=True,
        do_sample=False,  # greedy per benchmark riproducibile
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

torch.cuda.synchronize()
elapsed = time.time() - start

n_generated = generated_ids.shape[1] - inputs.input_ids.shape[1]
tokens_per_sec = n_generated / elapsed

print(f"Token generati: {n_generated}")
print(f"Tempo totale:   {elapsed:.2f} s")
print(f"Velocita':      {tokens_per_sec:.1f} token/s")

---

## Note

| Versione | VRAM richiesta | Nota |
|----------|---------------|------|
| 7B bfloat16 (questo notebook) | ~14 GB | T4 al limite, A100 consigliata |
| W8ASpike (quantizzata int8) | ~7 GB | T4 comoda, carica da `Abel2076/SpikingBrain-7B-W8ASpike` |
| Vision-Language 7B | ~14 GB | Carica da `sherry12334/SpikingBrain-7B-VL` |

Per la versione **W8ASpike** (piu' leggera), sostituisci `MODEL_ID` nella cella 4:
```python
MODEL_ID = "Abel2076/SpikingBrain-7B-W8ASpike"
```
e usa `sys.path.insert(0, f"{REPO_DIR}/W8ASpike")` nella cella 3.